In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [3]:
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
             model="groq:llama-3.3-70b-versatile",
             trigger=("messages",10),
             keep=("messages",4)
        )
    ]
)

In [5]:
config={"configurable":{"thread_id":"test-1"}}

questions = [
    "What is 25 × 18 + 120 ÷ 6?",
    "If 40% of a number is 80, what is the original number?",
    "What is the square root and cube root of 144?",
    "Solve: 2x + 5 = 17",
    "Solve: x² - 5x + 6 = 0",
    "A train moves at 60 km/h for 3 hours. What distance does it cover?",
    "Find the area of a triangle with base 10 cm and height 6 cm",
    "Using Pythagoras theorem, if a=3 and b=4, find c",
    "A number multiplied by 5 and increased by 20 gives 70. Find the number",
    "Find the perimeter and area of a rectangle with length 12 and breadth 8"
]


In [ ]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    response=agent.invoke(HumanMessage(content=q),config=config)

human in loop middleware

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def  read_email_tool(email_id:str)->str:
    """Mock function to read an email by its id"""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"email sent to {recipient} with subject {subject}"

In [ ]:
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[send_email_tool,read_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]

)

In [ ]:
config={"configurable":{"thread_id":"test_approve"}}
response=agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@gmail.com with subject 'hello'  and body 'how are you'")]},
    config=config
)

In [ ]:
#step 2 approve:
from langgraph.types import Command
if "__interrupt__" in response:
    print("pause! Approving...")
    response=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {response}")